# 🏎️ F1 Lap Time Prediction — Sandbox Notebook

This notebook is a **sandbox** that exercises the full ML pipeline by calling the `src/` modules directly.  
No ML logic is re-implemented here — every step delegates to the production code.

| Step | Module called |
|------|---------------|
| Load data | `src.load_data.load_raw_data` |
| Clean data | `src.clean_data.clean_dataframe` |
| Validate | `src.validate.validate_dataframe` |
| Feature engineering | `src.features.get_feature_preprocessor` |
| Train & select model | `src.train.train_model` |
| Evaluate | `src.evaluate.evaluate_model` |
| Inference | `src.infer.run_inference` |

## 0. Environment Setup

Ensure the repo root is on `sys.path` so `src` is importable when running the notebook from inside `notebooks/`.

In [1]:
import os
import sys
from pathlib import Path

# Add the repo root to sys.path so 'src' is importable
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")

Repo root: C:\Users\Naven\Desktop\IE\17 MLOPS\Git_Repository\1-mlops-kickoff-repo


## 1. Imports — from `src` only

In [2]:
# Standard library
import yaml

# Third-party
import pandas as pd

# Local — src modules (no ML logic re-implemented here)
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference

print("All src modules imported successfully.")

All src modules imported successfully.


## 2. Load Configuration

All paths, feature lists, and model settings come from `config.yaml` — no hardcoded values.

In [3]:
config_path = REPO_ROOT / "config.yaml"

with config_path.open("r") as f:
    cfg = yaml.safe_load(f)

# Extract key sections
data_cfg    = cfg["data"]
task_cfg    = cfg["task"]
feat_cfg    = cfg["features"]

RAW_PATH       = REPO_ROOT / data_cfg["raw_path"]
TARGET_COLUMN  = task_cfg["target_column"]
YEAR_COLUMN    = task_cfg["year_column"]
TEST_YEAR      = task_cfg["test_year"]
NUMERIC_COLS   = feat_cfg["numeric_passthrough"]
CATEGORICAL_COLS = feat_cfg["categorical_onehot"]

print(f"Target      : {TARGET_COLUMN}")
print(f"Test year   : {TEST_YEAR}")
print(f"Numeric cols: {NUMERIC_COLS}")
print(f"Cat cols    : {CATEGORICAL_COLS}")

Target      : milliseconds
Test year   : 2023
Numeric cols: ['lap', 'grid', 'Stint', 'TyreLife', 'TrackTemp', 'Humidity', 'Pressure', 'Rainfall', 'WindSpeed', 'WindDirection']
Cat cols    : ['round', 'name', 'constructorId', 'code', 'Compound', 'FreshTyre']


## 3. Load Raw Data

In [4]:
df_raw = load_raw_data(RAW_PATH)

print(f"Raw data shape: {df_raw.shape}")
df_raw.head()

Raw data shape: (69230, 29)


,raceId,driverId,lap,position_x,time,milliseconds,year,round,circuitId,name,...,TyreLife,FreshTyre,Stint,AirTemp,TrackTemp,Humidity,Pressure,Rainfall,WindSpeed,WindDirection
0,1052,830,1,1,1:58.245,118245,2021,1,3,Bahrain International Circuit,...,4.0,False,1.0,20.5,27.6,54.6,1014.9,0.0,0.9,11.0
1,1052,830,2,1,2:22.406,142406,2021,1,3,Bahrain International Circuit,...,5.0,False,1.0,20.5,27.6,54.6,1014.9,0.0,0.9,11.0
2,1052,830,3,1,2:38.001,158001,2021,1,3,Bahrain International Circuit,...,6.0,False,1.0,20.5,27.6,54.6,1014.9,0.0,0.9,11.0
3,1052,830,4,1,1:44.343,104343,2021,1,3,Bahrain International Circuit,...,7.0,False,1.0,20.5,27.6,54.6,1014.9,0.0,0.9,11.0
4,1052,830,5,1,1:44.629,104629,2021,1,3,Bahrain International Circuit,...,8.0,False,1.0,20.5,27.6,54.6,1014.9,0.0,0.9,11.0


In [17]:
# Quick overview of the raw data
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69230 entries, 0 to 69229
Data columns (total 29 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   raceId         69230 non-null  int64  
 1   driverId       69230 non-null  int64  
 2   lap            69230 non-null  int64  
 3   position_x     69230 non-null  int64  
 4   time           69230 non-null  object 
 5   milliseconds   69230 non-null  int64  
 6   year           69230 non-null  int64  
 7   round          69230 non-null  int64  
 8   circuitId      69230 non-null  int64  
 9   name           69230 non-null  object 
 10  position_y     69155 non-null  float64
 11  grid           69230 non-null  object 
 12  constructorId  69230 non-null  int64  
 13  statusId       69230 non-null  int64  
 14  status         69230 non-null  object 
 15  code           69230 non-null  object 
 16  Driver         69230 non-null  object 
 17  LapNumber      69230 non-null  int64  
 18  Compou

## 4. Clean Data

In [5]:
df_clean = clean_dataframe(df_raw, target_column=TARGET_COLUMN)

print(f"Cleaned data shape : {df_clean.shape}")
print(f"Rows removed       : {len(df_raw) - len(df_clean)}")
df_clean.head()

Cleaned data shape : (65783, 18)
Rows removed       : 3447


,lap,milliseconds,year,round,name,grid,constructorId,code,Compound,TyreLife,FreshTyre,Stint,TrackTemp,Humidity,Pressure,Rainfall,WindSpeed,WindDirection
0,1,118245,2021,1,Bahrain International Circuit,1,9,VER,MEDIUM,4.0,False,1.0,27.6,54.6,1014.9,0.0,0.9,11.0
3,4,104343,2021,1,Bahrain International Circuit,1,9,VER,MEDIUM,7.0,False,1.0,27.6,54.6,1014.9,0.0,0.9,11.0
4,5,104629,2021,1,Bahrain International Circuit,1,9,VER,MEDIUM,8.0,False,1.0,27.6,54.6,1014.9,0.0,0.9,11.0
5,6,95982,2021,1,Bahrain International Circuit,1,9,VER,MEDIUM,9.0,False,1.0,27.6,54.6,1014.9,0.0,0.9,11.0
6,7,95902,2021,1,Bahrain International Circuit,1,9,VER,MEDIUM,10.0,False,1.0,27.6,54.6,1014.9,0.0,0.9,11.0


In [6]:
# Check target distribution after cleaning
df_clean[TARGET_COLUMN].describe()

count     65783.000000
mean      90296.447988
std       12530.097806
min       66200.000000
25%       80851.000000
50%       88548.000000
75%       98994.000000
max      128354.000000
Name: milliseconds, dtype: float64

## 5. Validate Schema

In [7]:
required_columns = [TARGET_COLUMN] + list(NUMERIC_COLS) + list(CATEGORICAL_COLS)

is_valid = validate_dataframe(df_clean, required_columns=required_columns)
print(f"Validation passed: {is_valid}")

Validation passed: True


## 6. Train / Test Split (year-based)

In [8]:
df_train = df_clean[df_clean[YEAR_COLUMN] < TEST_YEAR].copy()
df_test  = df_clean[df_clean[YEAR_COLUMN] == TEST_YEAR].copy()

X_train = df_train.drop(columns=[TARGET_COLUMN])
y_train = df_train[TARGET_COLUMN]

X_test  = df_test.drop(columns=[TARGET_COLUMN])
y_test  = df_test[TARGET_COLUMN]

print(f"Train rows: {len(X_train):,}  (seasons before {TEST_YEAR})")
print(f"Test rows : {len(X_test):,}  ({TEST_YEAR} season)")

Train rows: 42,260  (seasons before 2023)
Test rows : 23,523  (2023 season)


## 7. Feature Engineering

In [9]:
preprocessor = get_feature_preprocessor(
    quantile_bin_cols=[],
    categorical_onehot_cols=list(CATEGORICAL_COLS),
    numeric_passthrough_cols=list(NUMERIC_COLS),
)

print("Preprocessor built (unfitted). Steps:")
for name, transformer, cols in preprocessor.transformers:
    print(f"  {name}: {transformer.__class__.__name__} → {cols}")

Preprocessor built (unfitted). Steps:
  num: StandardScaler → ['lap', 'grid', 'Stint', 'TyreLife', 'TrackTemp', 'Humidity', 'Pressure', 'Rainfall', 'WindSpeed', 'WindDirection']
  cat: OneHotEncoder → ['round', 'name', 'constructorId', 'code', 'Compound', 'FreshTyre']


## 8. Train Model

`train_model` runs both candidate models (Linear Regression + Random Forest) and returns the one with the lowest CV RMSE.

In [10]:
train_result = train_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=preprocessor,
    problem_type=task_cfg["problem_type"],
)

model          = train_result["selected_model"]
selected_name  = train_result["selected_name"]
selected_score = train_result["selected_score"]
candidate_metrics = train_result["candidate_metrics"]

print(f"Selected model : {selected_name}")
print(f"CV RMSE        : {selected_score:,.2f} ms")
print()
print("All candidate metrics:")
for name, metrics in candidate_metrics.items():
    print(f"  {name}: CV RMSE = {metrics.get('cv_rmse', 'N/A'):,.2f} ms")

Failed to initialize W&B. Continuing without it. Error: module 'wandb' has no attribute 'init'


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Selected model : linear_regression
CV RMSE        : 5,982.92 ms

All candidate metrics:
  linear_regression: CV RMSE = 5,982.92 ms
  random_forest: CV RMSE = 8,113.65 ms


## 9. Evaluate on Test Set

In [11]:
from sklearn.metrics import r2_score

test_rmse = evaluate_model(
    model,
    X_test=X_test,
    y_test=y_test,
    problem_type=task_cfg["problem_type"],
)

train_preds = model.predict(X_train)
test_preds  = model.predict(X_test)

train_r2 = r2_score(y_train, train_preds)
test_r2  = r2_score(y_test,  test_preds)

print(f"Test RMSE : {test_rmse:,.2f} ms")
print(f"Train R²  : {train_r2:.4f}")
print(f"Test R²   : {test_r2:.4f}")

c:\Users\Naven\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Test RMSE : 6,872.93 ms
Train R²  : 0.7838
Test R²   : 0.6695


c:\Users\Naven\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


## 10. Run Inference

In [12]:
df_predictions = run_inference(model, X_infer=X_test)

print(f"Predictions shape: {df_predictions.shape}")
df_predictions.head(10)

Predictions shape: (23523, 1)


c:\Users\Naven\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [1, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


,prediction
44844,106927.897551
44845,106789.198981
44846,106650.500411
44847,106511.801841
44848,106373.103271
44849,106234.404701
44850,106095.706131
44851,105957.007561
44852,105818.308991
44853,105679.610421


In [13]:
# Compare predictions vs actuals on the first 10 rows
comparison = pd.DataFrame({
    "actual_ms":    y_test.values[:10],
    "predicted_ms": df_predictions["prediction"].values[:10],
})
comparison["error_ms"] = comparison["predicted_ms"] - comparison["actual_ms"]
comparison

,actual_ms,predicted_ms,error_ms
0,99019,106927.897551,7908.897551
1,97974,106789.198981,8815.198981
2,98006,106650.500411,8644.500411
3,97976,106511.801841,8535.801841
4,98035,106373.103271,8338.103271
5,97986,106234.404701,8248.404701
6,98021,106095.706131,8074.706131
7,98154,105957.007561,7803.007561
8,98278,105818.308991,7540.308991
9,98369,105679.610421,7310.610421


## Summary

This notebook ran the full pipeline end-to-end using only `src/` module calls.  
To run the full pipeline with W&B logging and artifact saving, use:

```bash
python -m src.main
```